In [16]:
# ALL BASED ON PROOF OF 7.1 BENNETT ET AL !!!!!!!!!!

# get power of fundamental unit

K.<w> = QuadraticField(1153)

U = K.unit_group()
fundamental_unit = U.fundamental_units()[0]
print(fundamental_unit)
# Factor a specific integer/rational within the number field, CHECK THAT 23 SPLITS IN RING OF INTEGERS FOR 277,601,877 

q = K.primes_above(23)[0]
print(q)

for a in range(0, 11):
    F = q.residue_field()          # Create the residue field (GF(23))
    phi = F.reduction_map()        # Get the reduction homomorphism K -> GF(23)
    
    val_reduced = 2+phi(w)    # Reduce mod q
    unit_reduced = phi(fundamental_unit^a)
    if val_reduced == unit_reduced:
        print("a must be equal to: ", a)


3017890256875073*w - 102475040023072656
Fractional ideal (-2905747*w - 98667120)
a must be equal to:  2


In [ ]:
# CHECKING 23 SPLITTING 
k.<w> = QuadraticField(1153)
print(k.disc())
print(kronecker(k.disc(),23))

1153
1


In [21]:
# GENERATING AND SOLVING THUE EQN for prime d and power a (NOTE FOR POSITIVE PRIMES)
def generate_and_solve_thue(d, a, N=11, target=1):
    x = polygen(QQ)
    K.<w> = NumberField(x^2 - d)
    U_grp = K.unit_group()
    fundamental_unit = U_grp.fundamental_units()[0]
    unit_power = K(fundamental_unit)^a
    
    # Extract coefficients in the {1, sqrt(d)} basis safely (even when d % 4 == 1)
    coeffs = unit_power.list()
    c0 = QQ(coeffs[0])
    c1 = QQ(coeffs[1]) if len(coeffs) > 1 else QQ(0)
    
    # 1. Setup the polynomial ring for U, V over Rational Field
    R.<U, V> = PolynomialRing(QQ)
    
    # 2. Compute (U + V*sqrt(d))^N = A(U,V) + B(U,V)*sqrt(d)
    A = R(0)
    B = R(0)
    for k in range(N + 1):
        term = binomial(N, k) * (U^(N - k)) * (V^k)
        if k % 2 == 0:
            A += term * (d^(k // 2))
        else:
            B += term * (d^((k - 1) // 2))
            
    # 3. Extract the coefficient of sqrt(d)
    F = c0 * B + c1 * A
    
    # Reduce polynomial and capture the transformation matrix M
    F_min, M = F.reduced_form()

    # 4. Convert F_min(U', V') to univariate form for PARI/GP
    f_univariate = F_min.subs({V: 1})
    
    # 5. Solve using PARI's thue solver
    pari_sols = gp.thue(f_univariate, target)
    reduced_solutions = pari_sols.sage()
    
    # 6. Recover original (U, V) = M * [U', V']^T and verify
    original_solutions = []
    for sol in reduced_solutions:
        uv_orig = M * vector(ZZ, sol)
        u_orig, v_orig = uv_orig[0], uv_orig[1]
        
        # Sanity check: verify F(u_orig, v_orig) == target
        assert F(u_orig, v_orig) == target, f"Mismatch for {sol}"
        original_solutions.append([u_orig, v_orig])
        
    return F, original_solutions

generate_and_solve_thue(1153,2)

(-618516849717028248841100388607776*U^11 + 231024944210067539257986204007675403*U^10*V - 39223246024805346400258381143562115040*U^9*V^2 + 3995576410113118091466871398312746094885*U^8*V^3 - 271346415999603386396987480751162711846720*U^7*V^4 + 12899318882409190446491647622312869492726734*U^6*V^5 - 438007384706559786322017191428526849462975424*U^5*V^6 + 10623510479584140417717764077519098946509945930*U^4*V^7 - 180365183773808369153316364898961234796718094240*U^3*V^8 + 2041484597160085650271430330229920180887661276215*U^2*V^9 - 13864070459413403308918251248566820248041064177248*U*V^10 + 42796940736828704632053803104638144882972244572289*V^11,
 [])

In [ ]:
# FOR NEGATIVE t(p^2)!!!!
def solve_thue_fast(alpha_val):
    """
    Uses V | 2048 and U == V (mod 2) to reduce to V in {-2, -1, 1, 2},
    then checks for integer roots in U.
    """
    alpha_val = ZZ(alpha_val)
    R.<U> = PolynomialRing(ZZ)
    valid_sols = []

    for V_val in [-2, -1, 1, 2]:
        poly = sum(
            binomial(11, 2*k + 1) * U^(10 - 2*k) * (V_val^(2*k + 1)) * (alpha_val^k)
            for k in range(6)
        ) - 2048
        print(poly)

        for U_val, _ in poly.roots(ZZ):
            if (U_val - V_val) % 2 == 0:
                valid_sols.append((U_val, V_val))

    return valid_sols

solve_thue_fast(-503)

-22*U^10 + 663960*U^8 - 3740485056*U^6 + 5375611380480*U^4 - 1802621682920960*U^2 + 65943178655215616
-11*U^10 + 82995*U^8 - 116890158*U^6 + 41996963910*U^4 - 3520745474455*U^2 + 32198817700695
11*U^10 - 82995*U^8 + 116890158*U^6 - 41996963910*U^4 + 3520745474455*U^2 - 32198817704791
22*U^10 - 663960*U^8 + 3740485056*U^6 - 5375611380480*U^4 + 1802621682920960*U^2 - 65943178655219712


[]